<a href="https://colab.research.google.com/github/andiekobbietks/-autonomousagent/blob/main/studio/Unsloth_Studio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth Studio on your local device, follow [our guide](https://unsloth.ai/docs/new/unsloth-studio/install). Unsloth Studio is licensed [AGPL-3.0](https://github.com/unslothai/unsloth/blob/main/studio/LICENSE.AGPL-3.0).

### Unsloth Studio

Train and run open models with [**Unsloth Studio**](https://unsloth.ai/docs/new/unsloth-studio/start). NEW! Installation should now only take 2 mins!


We are actively working on making Unsloth Studio install on Colab T4 GPUs faster.

[Features](https://unsloth.ai/docs/new/unsloth-studio#features) • [Quickstart](https://unsloth.ai/docs/new/unsloth-studio/start) • [Data Recipes](https://unsloth.ai/docs/new/unsloth-studio/data-recipe) • [Studio Chat](https://unsloth.ai/docs/new/unsloth-studio/chat) • [Export](https://unsloth.ai/docs/new/unsloth-studio/export)

<p align="left"><img src="https://github.com/unslothai/unsloth/raw/main/studio/frontend/public/studio%20github%20landscape%20colab%20display.png" width="600"></p>

### Setup: Clone repo and run setup

In [ ]:
!git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git
%cd /content/unsloth
!chmod +x studio/setup.sh && ./studio/setup.sh --local

### Start Unsloth Studio

In [ ]:
import sys, time
sys.path.insert(0, "/content/unsloth/studio/backend")
from colab import start
start()

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8888, height = 1200, width = "100%")
for _ in range(10000): time.sleep(300), print("=", end = "")

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

from unsloth import FastLanguageModel
import torch

# 1. Load the model (Gemma-2-9b or similar)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-9b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Define the 'Probe' (Activation Hook)
activations = {}
def get_activation(name):
    def hook(model, input, output):
        # 'output' contains the hidden states for this layer
        activations[name] = output.detach()
    return hook

# 3. Attach the hook to a middle layer (e.g., layer 20)
layer_to_probe = model.model.layers[20]
handle = layer_to_probe.register_forward_hook(get_activation('layer_20'))

# 4. Run Inference
inputs = tokenizer(["Explain your internal goal state."], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 64)

# 5. Inspect the Latent Space
print("Extracted Latent Vector Shape:", activations['layer_20'].shape)
display(activations['layer_20'])

# Remove the hook
handle.remove()

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ch4r2yw1/unsloth_255461cfec0f49dbb7cd87d026787dd1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ch4r2yw1/unsloth_255461cfec0f49dbb7cd87d026787dd1
  Resolved https://github.com/unslothai/unsloth.git to commit d1725a31aacf001a052f7ce8be122470028da948
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.0 MB/s eta 0:00:0